In [ ]:
#Imports: 

import mlflow
import os
from datetime import date, datetime

#1. Configuração do caminho
current_path = os.getcwd()
if os.path.basename(current_path) == 'notebooks':
    project_root = os.path.abspath(os.path.join(current_path, '..'))
else:
    project_root = current_path

mlruns_path = os.path.join(project_root, "mlruns")
tracking_uri = "file:///" + mlruns_path.replace("\\", "/")

print(f"-> Conectando ao MLflow em: {tracking_uri}")
mlflow.set_tracking_uri(tracking_uri)

mlflow.set_experiment("Auditoria_Manutencao_PIBIC")

#2. Localização dos arquivos
docs_path = os.path.join(project_root, "docs", "nasa")
arquivo_html = os.path.join(docs_path, "1_Nasa_Feature_Importance.html")
arquivo_metricas = os.path.join(docs_path, "3_Nasa_Metricas.txt")

#3. Leitura das métricas
mae, rmse = 29.69, 41.52

if os.path.exists(arquivo_metricas):
    with open(arquivo_metricas, 'r', encoding='utf-8') as f:
        linhas = f.readlines()
        for linha in linhas:
            if "MAE" in linha:
                mae = float(linha.split(":")[1].strip())
            elif "RMSE" in linha:
                rmse = float(linha.split(":")[1].strip())

#4. Registro no ML Flow
print("Iniciando registro da Execução...")

with mlflow.start_run(run_name="Estudo_C_NASA_Turbofan"):
    
    #4.1 Logando os Parâmetros Técnicos
    mlflow.log_param("dataset", "NASA C-MAPSS FD001")
    mlflow.log_param("modelo", "Random Forest Regressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("tipo_problema", "Regressao_RUL (Vida Util)")
    
    #4.2 Logando as Métricas de Erro
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    
    #4.3 Logando a Documentação

    #Atualização automática de data:
    dia_atual = date.today()

    descricao = """
    Estudo de Caso C: Prognóstico Aeroespacial (Dados NASA)
    Objetivo: Prever a Vida Útil Restante (RUL - Remaining Useful Life) de motores Turbofan comerciais.
    Dataset: NASA C-MAPSS (Sub-dataset FD001).
    
    * Problema: Regressão contínua.
    * Métrica de Sucesso: MAE (Erro Médio Absoluto) de ~29 ciclos (voos).
    
    Insights de Explicabilidade (XAI - Shapash):
    O modelo operou sobre as leis da termodinâmica, elegendo a Pressão Estática na Saída do HPC (Sensor 11), o Sensor 9 e a Temperatura de Saída da Turbina (Sensor 4) como os principais preditores de falha iminente. 
    
    Isso comprova a robustez da arquitetura: o pipeline criado escala de sistemas de manufatura (Torque) para hidráulica (Eficiência) e até propulsão aeroespacial (Pressão Estática), mantendo total explicabilidade para o operador (Indústria 5.0).
    """
    mlflow.set_tag("mlflow.note.content", descricao)
    
    #4.4 Anexando os Gráficos do Shapash
    if os.path.exists(arquivo_html):
        mlflow.log_artifact(arquivo_html, artifact_path="xai_reports")
        print("Artefato HTML registrado com sucesso!")
    else:
        print("Arquivo HTML do Shapash não encontrado.")
        
    if os.path.exists(arquivo_metricas):
        mlflow.log_artifact(arquivo_metricas, artifact_path="metrics")

print("\n Experimento da NASA registrado no MLflow.")

-> Conectando ao MLflow em: file:///c:/XAILabProcess/mlruns
-> Iniciando registro da Execução (Run)...
-> Artefato HTML registrado com sucesso!

✅ TUDO PRONTO! O experimento da NASA foi eternizado no MLflow.
